# DIVRS — Reproduce trên Colab (ML-10M)

Code đã fix sẵn ở repo **[HatDuaa/DIVRS-reproduce](https://github.com/HatDuaa/DIVRS-reproduce)**. Notebook: **clone repo → tải data → chạy**. Không vá gì cả.

Bật GPU: **Runtime → Change runtime type → T4 GPU**.

## 1. GPU + lib

In [ ]:
!nvidia-smi
!pip install -q absl-py setproctitle deprecated faiss-cpu
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. Clone code (tự `git pull` nếu đã có — luôn lấy bản mới nhất)

In [ ]:
import os
REPO = '/content/DIVRS'
if os.path.exists(REPO):
    !cd {REPO} && git pull -q
else:
    !git clone -q https://github.com/HatDuaa/DIVRS-reproduce.git {REPO}
%cd {REPO}
!ls

## 3. Tải data ML-10M (đã tiền xử lý, format DICE)
Tải `ml10m.zip` (~42 MB) → giải nén ra `data/ml10m/output/` (đủ 6 file bắt buộc + 3 graph).

In [ ]:
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ml10m/output/train_coo_record.npz'):
    !cd data && wget -q https://raw.githubusercontent.com/tsinghua-fib-lab/DICE/main/data/ml10m.zip && unzip -oq ml10m.zip

need = ['train_coo_record.npz','val_coo_record.npz','test_coo_record.npz',
        'train_skew_coo_record.npz','popularity.npy','popularity_blend.npy']
miss = [f for f in need if not os.path.exists('data/ml10m/output/'+f)]
print('Data thieu:', miss if miss else 'KHONG — du, san sang!')

## 4. Train + Test — ML-10M / DIVRS (MF)
`epochs=30` chạy thử nhanh; bỏ flag đó để chạy đầy đủ (mặc định 500 + early-stop).

In [ ]:
!python app.py \
  --flagfile config/ml10m_DIVRS.cfg \
  --output ./output/ \
  --use_gpu=True --gpu_id=0 \
  --cg_use_gpu=False \
  --num_workers=2 \
  --epochs=30 --es_patience=3

## 5. Ghi chú
- **Baseline so sánh**: đổi `--flagfile` → `config/ml10m_mf.cfg` / `ml10m_ips.cfg` / `ml10m_cause.cfg` / `ml10m_lgn.cfg`.
- **GCN** (`config/ml10m_DIVRS-GCN.cfg`): cần cài `dgl` khớp torch; file `*_coo_adj_graph.npz` đã có sẵn trong data.
- **Douban**: chưa có data sẵn (phải tự build qua pipeline dps) — để sau.
- Kết quả Recall@K/NDCG in ra log; lưu ở `./output/`. Sửa code thì commit vào repo rồi cell 2 sẽ tự `git pull`.